In [ ]:
import sqlite3
import time

db = sqlite3.connect("orders.db")

db.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        id TEXT PRIMARY KEY,
        amount TEXT
    )
""")

def incoming_orders():
    yield (1, 100)
    yield (2, 200)
    yield (3, 300)

# Hardcoded application-level access control role
current_user_role = "admin"
AUTHORIZED_ROLES = {"admin", "system_service"}

def check_access(user_role):
    if user_role not in AUTHORIZED_ROLES:
        raise PermissionError(f"Access Denied: Role '{user_role}' is not authorized.")

processed_count = 0
failed_count = 0
start_time = time.time()

try:
    check_access(current_user_role)
    
    for order in incoming_orders():
        try:
            db.execute(
                "INSERT OR IGNORE INTO orders VALUES (?, ?)",
                order
            )
            db.commit()
            processed_count += 1
            print(f"[MONITOR] SUCCESS: Stored order {order}")
        except Exception as e:
            failed_count += 1
            print(f"[MONITOR] ERROR: Failed to store order {order}: {e}")

except PermissionError as auth_error:
    print(f"[SECURITY AUDIT] Unauthorized access attempt blocked: {auth_error}")

duration = time.time() - start_time
print("\n--- PIPELINE MONITORING & AUDIT REPORT ---")
print(f"User Role Checked: {current_user_role}")
print(f"Total Processed Successfully: {processed_count}")
print(f"Total Failed: {failed_count}")
print(f"Execution Time: {duration:.4f} seconds")

db.close()